In [1]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('ourall.csv')
# for i in df.columns:
#     if df[i].dtype is not np.float64:
#         df[i] = df[i].astype(np.float64)
print(df.columns)
df = df[df['B4TOD4'] != 0.005]
df = df[df['OMEGA5'] != 15]
df = df[df['OMEGA5'] != -15]
# scaler = MinMaxScaler()
# Fit and transform the data
# df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
scaler = MinMaxScaler()
# Fit and transform the data
df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
X_pool = df.drop("PT_LOSS", axis=1).values
y_pool = df["PT_LOSS"].values.reshape(-1, 1)
print(df['B4TOD4'].unique(),'B4TOD4')
print(df['B5TOB4'].unique(),'B5TOB4')
print(df['R45TOB4'].unique(),'R45TOB4')
print(df['OMEGA5'].unique(),'OMEGA5')
print(df['GAMMAU'].unique(),'GAMMAU')
print(df['ALPHA4'].unique(),'ALPHA4')
print(df['RE'].unique(),'RE')

print(len(y_pool))
print(X_pool.shape, y_pool.shape)
# print(X_pool,type(X_pool[100]))
# print(X_pool)
# tX, ty = torch.from_numpy(X_pool).float(), torch.from_numpy(y_pool).float()
# dataset = create_dataset_

Index(['B4TOD4', 'B5TOB4', 'R45TOB4', 'OMEGA5', 'GAMMAU', 'ALPHA4', 'RE',
       'PT_LOSS'],
      dtype='object')
[0.         0.42857143 0.71428571 1.        ] B4TOD4
[0.5 1.  0. ] B5TOB4
[0.        0.3902439 1.       ] R45TOB4
[0.  0.5 1. ] OMEGA5
[0.  1.  0.5] GAMMAU
[0.  0.2 0.4 0.6 0.8 1. ] ALPHA4
[0. 1.] RE
3888
(3888, 7) (3888, 1)


In [2]:
def NA_QBC(committee, X_sample):
    grads = []
    for i in range(X_sample.shape[0]):
        x_s = np.array([X_sample[i]])
        qbc_loss, tx = qbc(committee, x_s)
        if tx.grad is not None:
            tx.grad.zero_()
        qbc_loss.backward(retain_graph=True)
        grads.append(tx.grad.detach().clone())
    return grads

def get_steps(x):
    steps = [{-1:None, 0: 0 , 1: None} for i in range(7)]
# [0.01 0.04 0.06 0.08] B4TOD4
# [1.25 1.5  1.  ] B5TOB4
# [0.9 2.5 5. ] R45TOB4
# [-5  0  5] OMEGA5
# [-0.5  0.5  0. ] GAMMAU
# [10 20 30 40 50 60] ALPHA4
# [100000 500000] RE
    
# [0.         0.42857143 0.71428571 1.        ] B4TOD4
# [0.5 1.  0. ] B5TOB4
# [0.        0.3902439 1.       ] R45TOB4
# [0.  0.5 1. ] OMEGA5
# [0.  1.  0.5] GAMMAU
# [0.  0.2 0.4 0.6 0.8 1. ] ALPHA4
# [0. 1.] RE

    
    if x[0] <= 0.2:
        steps[0][1] = 0.42857143
        steps[0][-1] = 0
    elif x[0] <= 0.5:
        steps[0][1] = 0.71428571-0.42857143
        steps[0][-1] = -0.42857143
    elif x[0] <= 0.75:
        steps[0][1] = 1.0-0.71428571
        steps[0][-1] = -0.71428571-0.42857143
    elif x[0] > 0.72:
        steps[0][1] = 0
        steps[0][-1] = -(1.0-0.71428571)

    if x[1] <= 0.15:
        steps[1][1] = 0.5
        steps[1][-1] = 0
    elif x[1] <= 0.6:
        steps[1][1] = 0.5
        steps[1][-1] = -0.5
    elif x[1] > 0.7:
        steps[1][1] = 0
        steps[1][-1] = -0.5

    if x[2] <= 0.35:
        steps[2][1] = 0.3902439
        steps[2][-1] = 0
    elif x[2] <= 0.7:
        steps[2][1] = 1-0.3902439
        steps[2][-1] = -0.3902439
    elif x[2] > 0.7:
        steps[2][1] = 0
        steps[2][-1] = -(1-0.3902439)

    if x[3] <= 0.15:
        steps[3][1] = 0.5
        steps[3][-1] = 0
    elif x[3] <= 0.6:
        steps[3][1] = 0.5
        steps[3][-1] = -0.5
    elif x[3] > 0.7:
        steps[3][1] = 0
        steps[3][-1] = -0.5

    if x[4] <= 0.25:
        steps[4][1] = 0.5
        steps[4][-1] = 0
    elif x[4] <= 0.75:
        steps[4][1] = 0.5
        steps[4][-1] = -0.5
    elif x[4] > 0.7:
        steps[4][1] = 0
        steps[4][-1] = -0.5

    if x[5] <= 0.15:
        steps[5][1] = 0.2
        steps[5][-1] = 0
    elif x[5] <= 0.25:
        steps[5][1] = 0.2
        steps[5][-1] = -0.2
    elif x[5] <= 0.55:
        steps[5][1] = 0.2
        steps[5][-1] = -0.2
    elif x[5] <= 0.75:
        steps[5][1] = 0.2
        steps[5][-1] = -0.2
    elif x[5] <= 0.87:
        steps[5][1] = 0.2
        steps[5][-1] = -0.2
    elif x[5] > 0.95:
        steps[5][1] = 0
        steps[5][-1] = -0.2

    if x[6] <= 0.5:
        steps[6][1] = 1.
        steps[6][-1] = 0
    elif x[6] >= 0.6:
        steps[6][1] = 0.0
        steps[6][-1] = -1.
    return steps
        


# [0.005 0.02  0.1  ] B4TOD4
# [1.25 1.   1.5 ] B5TOB4
# [0.9 5.  2.5] R45TOB4
# [-5.  5.  0.] OMEGA5
# [-0.5  0.5  0. ] GAMMAU
# [10. 30. 60. 20. 40. 50.] ALPHA4
# [100000. 500000.] RE

def Lbnd(X_sample, min_max_val =[[0.005,0.1],[1.,1.5],[0.9,5.],[-5.,5.],[-0.5,0.5],[10.,60.],[100000.,500000.]]):
    
    l_grad_bnd = []
    for i in range(X_sample.shape[0]):
        l_grad = []
        x_s = np.array([X_sample[i]])
        for ind in range(len(x_s[0])):
            if  x_s[0][ind] >= min_max_val[ind][1]:
                l_grad.append(-1.0)
            elif min_max_val[ind][0] < x_s[0][ind] < min_max_val[ind][1]:
                l_grad.append(0.0)
            elif  x_s[0][ind] <= min_max_val[ind][0]:
                l_grad.append(1.0)
        l_grad_bnd.append(torch.tensor([l_grad]))
    return l_grad_bnd
    
def NA_query_strategy(comittee, X_sample):
    for i in range(4):
        grads = NA_QBC(comittee, X_sample) # (N, M) -> N
        l_grad_bnd = Lbnd(X_sample)
        result = [a - b for a, b in zip(grads, l_grad_bnd)]
        print(result, 'result grads - l_bnd')
        sign = [torch.where(tensor > 0.00001, torch.tensor(1), 
                  torch.where(tensor < 0, torch.tensor(-1), torch.tensor(0))) for tensor in result]
        # sign = [torch.where(tensor > 0, torch.tensor(1)), 
        #   torch.where(tensor < 0, torch.tensor(-1)), torch.where(tensor <= 0.00001, torch.tensor(0)) for tensor in result]
        print(sign, 'sign grads - l_bnd')
        for ind in range(len(sign)):
            real_steps = []
            sign_list = sign[ind].squeeze().tolist()
            steps = get_steps(X_sample[ind])
            x_gen = None
            for index, values in enumerate(sign_list):
                real_steps.append(steps[index][values])
            print(real_steps,'real_steps')
            x_gen = X_sample[ind] + real_steps
            X_sample[ind] = x_gen.copy()
        print(X_sample,"X_sample",i)
    return None, X_sample


def get_new_y(X_sampels,X_pool,y_pool):
    index_list = []
    X_study = []
    y_study = []
    for target_row in X_sampels:
        # print(X_sampels)
        print(target_row)
        # index = np.where((X_pool == target_row).all(axis=1))[0]
        index = np.where(np.all(np.isclose(X_pool, target_row), axis=1))[0]
        # index = np.where(np.all(X_pool == target_row, axis=1))[0]
        s=0
        if index.size > 0:
            # print(f"Найдена строка {target_row} на индексе {index[0]}")
            if index[0] not in index_list:
                X_study.append(target_row)
                y_study.append(y_pool[index[0]])
                index_list.append(index[0])
                X_pool = np.delete(X_pool, index[0], axis=0)
                y_pool = np.delete(y_pool, index[0])
                print(f"Найдена строка {target_row} на индексе {index[0]}")
            else:
                help_copy1 = target_row.copy()
                help_copy2 = target_row.copy()
                while s == 0:
                    i = 0 
                    step1 = get_steps(help_copy1)
                    step2 = get_steps(help_copy2)
                    print(step1,"step")
                    for i in range(7):
                        help_copy1[i] += step1[i][1]
                        help_copy2[i] += step2[i][-1]
                        index2 = np.where(np.all(np.isclose(X_pool, help_copy1), axis=1))[0]
                        if index2.size > 0 and index2[0] not in index_list:
                            s=1
                            print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                            X_study.append(help_copy1)
                            y_study.append(y_pool[index2[0]])
                            index_list.append(index2[0]) 
                            X_pool = np.delete(X_pool, index2[0], axis=0)
                            y_pool = np.delete(y_pool, index2[0])
                            break
                        index3 = np.where(np.all(np.isclose(X_pool, help_copy2), axis=1))[0]
                        if index3.size > 0 and index3[0] not in index_list:
                            s=1
                            print(f"Найдена1 строка {target_row} на индексе {index3[0]}")
                            X_study.append(help_copy2)
                            y_study.append(y_pool[index3[0]])
                            index_list.append(index3[0]) 
                            X_pool = np.delete(X_pool, index3[0], axis=0)
                            y_pool = np.delete(y_pool, index3[0])
                            break
                # print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                # X_study.append(help_copy)
                # y_study.append(y_pool[index2[0]])
                # index_list.append(index2[0]) 
                # X_pool = np.delete(X_pool, index2[0], axis=0)
                # y_pool = np.delete(y_pool, index2[0])
                        
                    
        else:
            help_copy1 = target_row.copy()
            help_copy2 = target_row.copy()
            while s == 0:
                i = 0 
                step1 = get_steps(help_copy1)
                step2 = get_steps(help_copy2)
                print(step1,"step1")
                for i in range(7):
                    help_copy1[i] += step1[i][1]
                    help_copy2[i] += step2[i][-1]
                    index2 = np.where(np.all(np.isclose(X_pool, help_copy1), axis=1))[0]
                    if index2.size > 0 and index2[0] not in index_list:
                        s=1
                        print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                        X_study.append(help_copy1)
                        y_study.append(y_pool[index2[0]])
                        index_list.append(index2[0]) 
                        X_pool = np.delete(X_pool, index2[0], axis=0)
                        y_pool = np.delete(y_pool, index2[0])
                        break
                    index3 = np.where(np.all(np.isclose(X_pool, help_copy2), axis=1))[0]
                    if index3.size > 0 and index3[0] not in index_list:
                        s=1
                        print(f"Найдена1 строка {target_row} на индексе {index3[0]}")
                        X_study.append(help_copy2)
                        y_study.append(y_pool[index3[0]])
                        index_list.append(index3[0]) 
                        X_pool = np.delete(X_pool, index3[0], axis=0)
                        y_pool = np.delete(y_pool, index3[0])
                        break
            
    return X_study, y_study, index_list

In [3]:
from typing import Optional  # Добавлено
import torch
import gpytorch
import numpy as np
from gpytorch.models import ExactGP
from gpytorch.means import ConstantMean
from gpytorch.kernels import RBFKernel, MaternKernel, ScaleKernel, PeriodicKernel  # Добавлены ScaleKernel и PeriodicKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.constraints import Interval
from sklearn.base import BaseEstimator  # Добавлено
from modAL.models import ActiveLearner  # Добавлено, если используется modAL
from sklearn.metrics import mean_absolute_error,r2_score

class GPModel(ExactGP):
    def __init__(
        self,
        train_x: Optional[torch.Tensor] = None,
        train_y: Optional[torch.Tensor] = None,
        likelihood: Optional[GaussianLikelihood] = None,
        kernel_type: str = 'matern',
        nu: float = 0.5,
        lengthscale_constraint: Optional[Interval] = None,
        outputscale_constraint: Optional[Interval] = None
    ):
        """
        Улучшенная GP модель с поддержкой разных ядер и modAL
        
        Параметры:
        ----------
        kernel_type : str
            Тип ядра ('matern', 'rbf', 'periodic')
        nu : float
            Параметр гладкости для ядра Matern
        """
        if likelihood is None:
            likelihood = GaussianLikelihood()
        
        super().__init__(train_x, train_y, likelihood)
        
        # Средняя функция
        self.mean_module = ConstantMean()
        
        # Инициализация ограничений
        if lengthscale_constraint is None:
            lengthscale_constraint = Interval(1e-5, 1e5)  # Изменено на Interval
        if outputscale_constraint is None:
            outputscale_constraint = Interval(1e-5, 1e5)  # Изменено на Interval
        
        # Выбор ядра
        self.kernel_type = kernel_type.lower()
        if self.kernel_type == 'matern':
            base_kernel = MaternKernel(
                nu=nu,
                lengthscale_constraint=lengthscale_constraint
            )
        elif self.kernel_type == 'rbf':
            base_kernel = RBFKernel(
                lengthscale_constraint=lengthscale_constraint
            )
        elif self.kernel_type == 'periodic':
            base_kernel = PeriodicKernel(
                lengthscale_constraint=lengthscale_constraint
            )
        else:
            raise ValueError(f"Unknown kernel type: {kernel_type}. Supported: 'matern', 'rbf', 'periodic'")
        
        self.covar_module = ScaleKernel(
            base_kernel,
            outputscale_constraint=outputscale_constraint
        )
        
        self.initialize_parameters()
    
    def initialize_parameters(self):
        """Инициализация параметров модели"""
        if self.train_inputs is not None:
            init_lengthscale = self.train_inputs[0].std().item()
            self.covar_module.base_kernel.lengthscale = init_lengthscale
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)
    
    def fit(self, train_x, train_y, n_iters=100, lr=0.1, verbose=True):
        # Преобразуем данные в тензоры, если это ещё не сделано
        if not isinstance(train_x, torch.Tensor):
            train_x = torch.tensor(train_x, dtype=torch.float32)
        if not isinstance(train_y, torch.Tensor):
            train_y = torch.tensor(train_y, dtype=torch.float32)
    
        # Убедимся, что y — одномерный тензор (иначе squeeze)
        if len(train_y.shape) > 1:
            train_y = train_y.squeeze(-1)
    
        self.set_train_data(train_x, train_y, strict=False)
        self.train()
        self.likelihood.train()
    
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        mll = gpytorch.mlls.ExactMarginalLogLikelihood(self.likelihood, self)
    
        for i in range(n_iters):
            optimizer.zero_grad()
            output = self(train_x)
            loss = -mll(output, train_y)
    
            # Если loss не скаляр (например, из-за batched данных), берём сумму
            if loss.dim() > 0:
                loss = loss.sum()
    
            loss.backward()  # Теперь градиенты вычислятся корректно
            optimizer.step()
    
            if verbose and (i % 10 == 0 or i == n_iters - 1):
                print(f'Iter {i+1}/{n_iters} - Loss: {loss.item():.3f}')
            
    def predict(self, test_x, return_std=True):
        """
        Предсказание (совместимо с modAL)
        """
        if not isinstance(test_x, torch.Tensor):
            test_x = torch.tensor(test_x, dtype=torch.float32)
            
        self.eval()
        self.likelihood.eval()
        
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            observed_pred = self.likelihood(self(test_x))
            if return_std:
                return observed_pred.mean.numpy(), observed_pred.stddev.numpy()
            return observed_pred.mean.numpy()
    
    def score(self, X, y):
        """
        Метод для совместимости с modAL (возвращает negative MSE)
        """
        pred, _ = self.predict(X)
        return -np.mean((pred - y)**2).item()


class GPModelSklearnWrapper(BaseEstimator):
    """Обертка для совместимости с sklearn/modAL"""
    def __init__(self, kernel_type='matern', nu=0.5, n_iters=100, lr=0.1):
        self.kernel_type = kernel_type
        self.nu = nu
        self.n_iters = n_iters
        self.lr = lr
        self.model = None
        
    def fit(self, X, y):
        self.model = GPModel(
            kernel_type=self.kernel_type,
            nu=self.nu
        )
        self.model.fit(X, y, n_iters=self.n_iters, lr=self.lr)
        return self
    
    def predict(self, X, return_std=False):
        if self.model is None:
            raise RuntimeError("Model not trained yet")
        return self.model.predict(X, return_std=return_std)
    
    def score(self, X, y):
        return self.model.score(X, y)


# Пример использования:
if __name__ == "__main__":
    # 1. Генерация данных
    np.random.seed(42)
    # X_pool = np.random.rand(100, 1)
    # y_pool = np.sin(X_pool * 2 * np.pi).ravel() + np.random.normal(0, 0.1, size=100)
    
    # 2. Подготовка данных
    train_idx = np.random.choice(range(X_pool.shape[0]), size=10, replace=False)
    train_x = X_pool[train_idx]
    train_y = y_pool[train_idx]
    
    test_idx = np.random.choice(range(X_pool.shape[0]), size=1, replace=False)
    test_x = X_pool[test_idx]
    test_y = y_pool[test_idx]
    # 3. Обучение модели
    model = GPModelSklearnWrapper(kernel_type='matern', nu=0.5)
    model.fit(train_x, train_y)
    
    # 4. Предсказание
    pred_mean, pred_std = model.predict(test_x, return_std=True)
    # print('mae',mean_absolute_error(y_test,pred_mean))
    # print('r2',r2_score(y_test,pred_mean))
    print("Predictions:", pred_mean)
    print("Uncertainty:", pred_std)

Iter 1/100 - Loss: 1.891
Iter 11/100 - Loss: 1.455
Iter 21/100 - Loss: 0.993
Iter 31/100 - Loss: 0.545
Iter 41/100 - Loss: 0.222
Iter 51/100 - Loss: 0.150
Iter 61/100 - Loss: 0.163
Iter 71/100 - Loss: 0.142
Iter 81/100 - Loss: 0.146
Iter 91/100 - Loss: 0.140
Iter 100/100 - Loss: 0.136
Predictions: [0.23357476]
Uncertainty: [0.14603269]


In [10]:
def create_active_learner(X_init, y_init, kernel_type='matern', nu = 0.5):
    # Создаем GP модель
    gp_model = GPModelSklearnWrapper(kernel_type=kernel_type,nu = nu)
    
    # Создаем ActiveLearner
    learner = ActiveLearner(
        estimator=gp_model,
        X_training=X_init,
        y_training=y_init,
        # query_strategy=your_query_strategy  # Например, uncertainty_sampling
    )
    return learner




def qbc(committee, X_sample):
    # Создаем тензор с отслеживанием градиентов
    tx = torch.tensor(X_sample, dtype=torch.float32, requires_grad=True)
    
    # Собираем предсказания
    preds = []
    for learner in committee.learner_list:
        model = learner.estimator
        model.train()  # Важно для вычисления градиентов!
        
        # Получаем предсказание (сохраняем граф вычислений)
        output = model(tx)
        pred = output.mean
        preds.append(pred)
    
    # Вычисляем QBC loss
    preds_tensor = torch.stack(preds)
    f_avg = preds_tensor.mean(dim=0)
    loss = torch.var(preds_tensor - f_avg, dim=0).mean()
    
    return loss, tx